In [66]:
!pip install seaborn scipy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

 1. LOAD DATA



In [67]:
teachers = pd.read_csv("/content/EduPro Online Platform.xlsx - Teachers.csv")
courses = pd.read_csv("/content/EduPro Online Platform.xlsx - Courses.csv")
transactions = pd.read_csv("/content/EduPro Online Platform.xlsx - Transactions.csv")
users = pd.read_csv("/content/EduPro Online Platform.xlsx - Users (1).csv")

print("Teachers:", teachers.shape)
print("Courses:", courses.shape)
print("Transactions:", transactions.shape)
print("Users:", users.shape)

Teachers: (60, 7)
Courses: (60, 8)
Transactions: (10000, 7)
Users: (3000, 5)


2. DATA INTEGRATION (Teachers <-> Transactions <-> Courses)

In [68]:
# Transactions is the bridge table: one row per enrollment/purchase,
# already carrying both TeacherID and CourseID.
df = transactions.merge(teachers, on="TeacherID", how="left")
df = df.merge(courses, on="CourseID", how="left")

print("\nMerged dataset shape:", df.shape)
print(df.head())

# Validate mapping: any transaction where teacher isn't actually linked to
# that course elsewhere, or nulls after merge
missing_teacher = df["TeacherName"].isna().sum()
missing_course = df["CourseName"].isna().sum()
print(f"\nUnmatched teacher rows: {missing_teacher}")
print(f"Unmatched course rows: {missing_course}")



Merged dataset shape: (10000, 20)
  TransactionID  UserID CourseID TransactionDate  Amount  PaymentMethod  \
0       TT00001  U00003  CR00016      25/10/2025     0.0         PayPal   
1       TT00002  U00003  CR00037       13/1/2025     0.0         PayPal   
2       TT00003  U00003  CR00019       28/3/2025     0.0  Bank Transfer   
3       TT00004  U00004  CR00048        2/6/2025     0.0  Bank Transfer   
4       TT00005  U00004  CR00060       10/8/2025     0.0         PayPal   

  TeacherID      TeacherName  Age  Gender         Expertise  \
0   TC00040  Kimberly Miller   49    Male     Cybersecurity   
1   TC00040  Kimberly Miller   49    Male     Cybersecurity   
2   TC00040  Kimberly Miller   49    Male     Cybersecurity   
3   TC00040  Kimberly Miller   49    Male     Cybersecurity   
4   TC00042   Yolanda Levine   49  Female  Machine Learning   

   YearsOfExperience  TeacherRating         CourseName  \
0                 24           4.58  Digital Marketing   
1                 2

3. INSTRUCTOR PROFILE ANALYSIS

In [69]:
print("\n--- Teacher Rating Distribution ---")
print(teachers["TeacherRating"].describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(teachers["Age"], bins=15, ax=axes[0], kde=True)
axes[0].set_title("Instructor Age Distribution")
sns.histplot(teachers["YearsOfExperience"], bins=15, ax=axes[1], kde=True)
axes[1].set_title("Instructor Experience Distribution")
sns.histplot(teachers["TeacherRating"], bins=15, ax=axes[2], kde=True)
axes[2].set_title("Instructor Rating Distribution")
plt.tight_layout()
plt.savefig("01_instructor_profile.png", dpi=150)
plt.close()


--- Teacher Rating Distribution ---
count    60.000000
mean      3.125000
std       0.949797
min       1.050000
25%       2.487500
50%       3.275000
75%       3.827500
max       4.970000
Name: TeacherRating, dtype: float64


 Top & Bottom Instructors

In [70]:
top_teachers = teachers.sort_values("TeacherRating", ascending=False).head(10)
bottom_teachers = teachers.sort_values("TeacherRating", ascending=True).head(10)

print("TOP 10:")
print(top_teachers[["TeacherName", "Expertise", "TeacherRating"]])
print("\nBOTTOM 10:")
print(bottom_teachers[["TeacherName", "Expertise", "TeacherRating"]])

TOP 10:
        TeacherName                Expertise  TeacherRating
41   Yolanda Levine         Machine Learning           4.97
39  Kimberly Miller            Cybersecurity           4.58
3      Kristi Scott         Machine Learning           4.39
36    Debra Escobar                  Finance           4.39
6      Angela Beard         Machine Learning           4.36
52      Aaron Kirby                Marketing           4.29
17      Kari Pierce       Project Management           4.28
53   Shelby Patrick  Artificial Intelligence           4.28
23      Michael Kim        Digital Marketing           4.22
1          Jill Day        Digital Marketing           4.14

BOTTOM 10:
        TeacherName                Expertise  TeacherRating
42    John Williams       Project Management           1.05
27   Timothy Bailey                 Business           1.06
44  Emily Gutierrez          Web Development           1.07
35    Brenda Mclean        Digital Marketing           1.39
26   Laurie Michael 

4. EXPERIENCE VS PERFORMANCE ANALYSIS

In [71]:
corr_exp_teacher, p1 = stats.pearsonr(teachers["YearsOfExperience"], teachers["TeacherRating"])
print(f"\nCorrelation (YearsOfExperience vs TeacherRating): r={corr_exp_teacher:.3f}, p={p1:.4f}")

# Bring TeacherRating + YearsOfExperience onto courses via merged df, avg per course
course_level_exp = df.groupby("CourseID").agg(
    CourseRating=("CourseRating", "first"),
    YearsOfExperience=("YearsOfExperience", "first")
).dropna()
corr_exp_course, p2 = stats.pearsonr(course_level_exp["YearsOfExperience"], course_level_exp["CourseRating"])
print(f"Correlation (YearsOfExperience vs CourseRating): r={corr_exp_course:.3f}, p={p2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(x="YearsOfExperience", y="TeacherRating", data=teachers, ax=axes[0],
            scatter_kws={"alpha": 0.6})
axes[0].set_title(f"Experience vs Teacher Rating (r={corr_exp_teacher:.2f})")

sns.regplot(x="YearsOfExperience", y="CourseRating", data=course_level_exp, ax=axes[1],
            scatter_kws={"alpha": 0.5}, color="orange")
axes[1].set_title(f"Experience vs Course Rating (r={corr_exp_course:.2f})")
plt.tight_layout()
plt.savefig("02_experience_vs_performance.png", dpi=150)
plt.close()

# Experience buckets to spot diminishing returns / thresholds
teachers["ExperienceBand"] = pd.cut(
    teachers["YearsOfExperience"],
    bins=[0, 2, 5, 8, 12, 100],
    labels=["0-2 yrs", "3-5 yrs", "6-8 yrs", "9-12 yrs", "12+ yrs"]
)
exp_band_summary = teachers.groupby("ExperienceBand")["TeacherRating"].mean()
print("\nAverage TeacherRating by Experience Band:\n", exp_band_summary)




Correlation (YearsOfExperience vs TeacherRating): r=0.598, p=0.0000
Correlation (YearsOfExperience vs CourseRating): r=0.010, p=0.9375

Average TeacherRating by Experience Band:
 ExperienceBand
0-2 yrs     2.643846
3-5 yrs     2.186667
6-8 yrs     3.626667
9-12 yrs    3.920000
12+ yrs     3.954000
Name: TeacherRating, dtype: float64


/tmp/ipykernel_4739/2344862367.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  exp_band_summary = teachers.groupby("ExperienceBand")["TeacherRating"].mean()


5. COURSE QUALITY EVALUATION

In [72]:
cat_rating = courses.groupby("CourseCategory")["CourseRating"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
print("\nCourse Rating by Category:\n", cat_rating)

level_rating = courses.groupby("CourseLevel")["CourseRating"].agg(["mean", "std", "count"])
print("\nCourse Rating by Level:\n", level_rating)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_rating["mean"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Average Course Rating by Category")
axes[0].set_ylabel("Avg Rating")
axes[0].tick_params(axis="x", rotation=45)

level_rating["mean"].plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Average Course Rating by Level")
axes[1].set_ylabel("Avg Rating")
plt.tight_layout()
plt.savefig("03_course_quality.png", dpi=150)
plt.close()

# Gender vs course level comparison (instructor gender, via merged df)
gender_level = df.groupby(["Gender", "CourseLevel"])["CourseRating"].mean().unstack()
print("\nAvg Course Rating: Instructor Gender x Course Level\n", gender_level)

plt.figure(figsize=(8, 5))
sns.heatmap(gender_level, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Course Rating: Instructor Gender vs Course Level")
plt.tight_layout()
plt.savefig("04_gender_level_heatmap.png", dpi=150)
plt.close()


Course Rating by Category:
                           mean       std  count
CourseCategory                                 
Marketing                3.722  1.200904      5
Digital Marketing        3.656  1.028557      5
Data Science             3.316  0.504014      5
Design                   3.162  1.291538      5
Project Management       3.142  1.707533      5
Programming              3.036  1.326774      5
Artificial Intelligence  3.036  1.240979      5
Finance                  3.010  1.100068      5
Cybersecurity            2.902  1.489587      5
Web Development          2.846  1.289411      5
Business                 2.688  0.728540      5
Machine Learning         2.658  1.559333      5

Course Rating by Level:
                   mean       std  count
CourseLevel                            
Advanced      2.813810  1.098902     21
Beginner      3.189048  1.299911     21
Intermediate  3.322778  1.090293     18

Avg Course Rating: Instructor Gender x Course Level
 CourseLevel  Advanc

6. INSTRUCTOR IMPACT ON COURSE SUCCESS (rating tiers)

In [73]:
teachers["RatingTier"] = pd.cut(
    teachers["TeacherRating"],
    bins=[0, 2.5, 3.75, 5],
    labels=["Low", "Mid", "High"]
)

df_tier = df.merge(teachers[["TeacherID", "RatingTier"]], on="TeacherID", how="left", suffixes=("", "_dup"))
# df already has RatingTier merged in via teachers join earlier? No - teachers merged before tier assigned.
# Recompute cleanly:
df = df.drop(columns=[c for c in df.columns if c.endswith("_dup")], errors="ignore")
tier_map = teachers.set_index("TeacherID")["RatingTier"]
df["RatingTier"] = df["TeacherID"].map(tier_map)

tier_summary = df.groupby("RatingTier").agg(
    AvgCourseRating=("CourseRating", "mean"),
    Enrollments=("TransactionID", "count")
)
print("\nInstructor Rating Tier -> Course Rating & Enrollments:\n", tier_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
tier_summary["AvgCourseRating"].plot(kind="bar", ax=axes[0], color="seagreen")
axes[0].set_title("Avg Course Rating by Instructor Tier")
tier_summary["Enrollments"].plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Enrollment Volume by Instructor Tier")
plt.tight_layout()
plt.savefig("05_instructor_tier_impact.png", dpi=150)
plt.close()

# ---------------------------------------------------------------
# 7. EXPERTISE-BASED PERFORMANCE INSIGHTS
# ---------------------------------------------------------------
expertise_summary = teachers.groupby("Expertise").agg(
    AvgTeacherRating=("TeacherRating", "mean"),
    AvgExperience=("YearsOfExperience", "mean"),
    Count=("TeacherID", "count")
).sort_values("AvgTeacherRating", ascending=False)
print("\nExpertise-wise Instructor Performance:\n", expertise_summary)

plt.figure(figsize=(9, 5))
expertise_summary["AvgTeacherRating"].plot(kind="barh", color="mediumpurple")
plt.title("Average Teacher Rating by Expertise Area")
plt.xlabel("Avg Rating")
plt.tight_layout()
plt.savefig("06_expertise_performance.png", dpi=150)
plt.close()

# ---------------------------------------------------------------
# 8. KPI CALCULATI

/tmp/ipykernel_4739/1741715508.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_summary = df.groupby("RatingTier").agg(



Instructor Rating Tier -> Course Rating & Enrollments:
             AvgCourseRating  Enrollments
RatingTier                              
Low                3.120008         1222
Mid                3.128024         1680
High               3.122716         7098

Expertise-wise Instructor Performance:
                          AvgTeacherRating  AvgExperience  Count
Expertise                                                      
Marketing                        4.290000       9.000000      1
Machine Learning                 4.234000       8.800000      5
Programming                      3.950000       9.000000      1
Cybersecurity                    3.468889       8.000000      9
Finance                          3.265000       5.333333      6
Artificial Intelligence          3.173333       5.500000      6
Digital Marketing                3.050000       5.363636     11
Data Science                     3.030000       6.200000      5
Project Management               2.664000       6.200000 

 7. EXPERTISE-BASED PERFORMANCE INSIGHTS

In [74]:
expertise_summary = teachers.groupby("Expertise").agg(
    AvgTeacherRating=("TeacherRating", "mean"),
    AvgExperience=("YearsOfExperience", "mean"),
    Count=("TeacherID", "count")
).sort_values("AvgTeacherRating", ascending=False)
print("\nExpertise-wise Instructor Performance:\n", expertise_summary)

plt.figure(figsize=(9, 5))
expertise_summary["AvgTeacherRating"].plot(kind="barh", color="mediumpurple")
plt.title("Average Teacher Rating by Expertise Area")
plt.xlabel("Avg Rating")
plt.tight_layout()
plt.savefig("06_expertise_performance.png", dpi=150)
plt.close()



Expertise-wise Instructor Performance:
                          AvgTeacherRating  AvgExperience  Count
Expertise                                                      
Marketing                        4.290000       9.000000      1
Machine Learning                 4.234000       8.800000      5
Programming                      3.950000       9.000000      1
Cybersecurity                    3.468889       8.000000      9
Finance                          3.265000       5.333333      6
Artificial Intelligence          3.173333       5.500000      6
Digital Marketing                3.050000       5.363636     11
Data Science                     3.030000       6.200000      5
Project Management               2.664000       6.200000      5
Web Development                  2.514000       4.000000      5
Design                           2.290000       5.500000      4
Business                         2.245000       7.500000      2


 KPI CALCULATIONS

In [75]:
avg_teacher_rating = teachers["TeacherRating"].mean()
avg_course_rating = courses["CourseRating"].mean()

# Rating Consistency Index: lower std = more consistent. Report as 1 - normalized std.
rating_std = teachers["TeacherRating"].std()
rating_consistency_index = 1 - (rating_std / teachers["TeacherRating"].mean())

# Experience Impact Score: correlation coefficient scaled 0-100
experience_impact_score = corr_exp_teacher * 100

# Enrollment Influence Ratio: enrollments of High tier / enrollments of Low tier
high_enroll = tier_summary.loc["High", "Enrollments"] if "High" in tier_summary.index else np.nan
low_enroll = tier_summary.loc["Low", "Enrollments"] if "Low" in tier_summary.index else np.nan
enrollment_influence_ratio = high_enroll / low_enroll if low_enroll else np.nan

print("\n===================== KPI SUMMARY =====================")
print(f"Average Teacher Rating:      {avg_teacher_rating:.2f}")
print(f"Average Course Rating:       {avg_course_rating:.2f}")
print(f"Rating Consistency Index:    {rating_consistency_index:.2f}")
print(f"Experience Impact Score:     {experience_impact_score:.1f}")
print(f"Enrollment Influence Ratio:  {enrollment_influence_ratio:.2f}")
print("========================================================")

kpi_df = pd.DataFrame({
    "KPI": ["Average Teacher Rating", "Average Course Rating", "Rating Consistency Index",
            "Experience Impact Score", "Enrollment Influence Ratio"],
    "Value": [round(avg_teacher_rating, 2), round(avg_course_rating, 2),
              round(rating_consistency_index, 2), round(experience_impact_score, 1),
              round(enrollment_influence_ratio, 2)]
})
kpi_df.to_csv("kpi_summary.csv", index=False)

# Save the merged master dataset for reuse in the Streamlit app
df.to_csv("merged_master.csv", index=False)

print("\nAll charts saved as PNGs, KPI summary saved to kpi_summary.csv,")
print("merged dataset saved to merged_master.csv")


===================== KPI SUMMARY =====================
Average Teacher Rating:      3.12
Average Course Rating:       3.10
Rating Consistency Index:    0.70
Experience Impact Score:     59.8
Enrollment Influence Ratio:  5.81

All charts saved as PNGs, KPI summary saved to kpi_summary.csv,
merged dataset saved to merged_master.csv
